# Challenge 9 · Protect the Logical Qubit
**Tier 3 - Advanced Challenges** · **Estimated time: 50-80 minutes** (an estimate, not a guarantee) · *Participant notebook*

> A memory qubit sits idle and an occasional bit flip sneaks in. Spread one logical qubit over three physical ones, detect the flip without looking at the data, and undo it.

### Learning objectives
- Encode a logical qubit in the three-qubit bit-flip repetition code.
- Extract an error syndrome without disturbing the encoded information.
- Decode the syndrome and apply the correction with classical feed-forward.
- State precisely what this code can and cannot protect against.

### Prerequisites
- Challenges 1-4 or equivalent
- CNOT and Pauli X
- Mid-circuit measurement and `if_test` (introduced in the notebook)

### What you will write
| Function | Status |
|---|---|
| `encode_circuit` | required |
| `syndrome_circuit` | required |
| `decode_syndrome` | required |
| `correction_circuit` | required |
| `add_feedforward_correction` | required |
| `phase_flip_encode_circuit` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `encode_circuit` maps alpha|0>+beta|1> (on qubit 0) to alpha|000>+beta|111>.
- `syndrome_circuit` writes the two parity checks into the ancillas without disturbing the logical information.
- `decode_syndrome` identifies each single bit-flip location, and 'no error', from the syndrome.
- `correction_circuit` applies X to the requested qubit (or nothing).
- `add_feedforward_correction` recovers the logical state after any single bit flip, for many logical states.
- Invalid syndromes and qubit indices raise ValueError.

**Bonus (optional, never required to pass):**
- `phase_flip_encode_circuit` yields a code that corrects any single Z error.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
A memory qubit sits idle and now and then a **bit flip** (an X error) sneaks in. You will protect **one logical qubit** with the **three-qubit repetition code**: spread it over three physical qubits, **detect** a flip without ever looking at the data, **decode** the evidence and **undo** the error with classical feed-forward.

1. `encode_circuit` - α|0⟩ + β|1⟩ ↦ α|000⟩ + β|111⟩.
2. `syndrome_circuit` - write two parity checks into two ancilla qubits.
3. `decode_syndrome` - from the two syndrome bits to the qubit that was flipped.
4. `correction_circuit` - the gate that undoes the flip.
5. `add_feedforward_correction` - the whole recovery step as a *dynamic circuit*: measure the ancillas, apply X controlled by the classical result.
6. *(Bonus)* `phase_flip_encode_circuit` - the same idea for phase flips.

### Layout and conventions (used by the notebook and by every test)
* **Data qubits** 0, 1, 2 hold the code word; **ancilla qubits** 3 and 4 start in |0⟩. The logical state is placed on qubit 0 *before* encoding; qubits 1 and 2 start in |0⟩.
* **Code words:** |0_L⟩ = |000⟩ and |1_L⟩ = |111⟩, so the encoded state of α|0⟩ + β|1⟩ is α|000⟩ + β|111⟩ (an entangled state - the qubit is *spread out*, not copied).
* **Syndrome** `(s0, s1)`: `s0` is the parity of data qubits 0 and 1 (stored in ancilla qubit 3), `s1` the parity of data qubits 1 and 2 (stored in ancilla qubit 4); the parity is 0 if the two bits agree and 1 if they differ.
* **Feed-forward:** a 2-bit classical register `syn` is passed to `add_feedforward_correction`; `syn[0]` receives the measurement of ancilla 0 (= `s0`), `syn[1]` that of ancilla 1 (= `s1`). The register holds the integer `s0 + 2·s1`, and conditions are written `with qc.if_test((syn, value)):` (a register or a single `Clbit` as the first element of the tuple; expression-based conditions are not supported by the exact simulator).
* All simulations in the tests are **exact** (every measurement outcome is enumerated, no sampling); tolerance 1e-9.

### Function contracts
The helper `validate_layout(qc, data, ancillas, syndrome)` is **provided**: it raises `ValueError` for an invalid layout and returns clean lists.

| Function | Input | Output | Raises `ValueError` when |
|---|---|---|---|
| `encode_circuit()` | - | 3-qubit circuit, unitary (no measurements), mapping (α\|0⟩ + β\|1⟩) ⊗ \|00⟩ to α\|000⟩ + β\|111⟩ for every α, β | - |
| `syndrome_circuit()` | - | 5-qubit circuit (data 0-2, ancillas 3-4), unitary: for ancillas in \|00⟩ it leaves the data unchanged and writes `s0` into qubit 3 and `s1` into qubit 4, for every basis state of the data qubits | - |
| `decode_syndrome(syndrome)` | two 0/1 integers (tuple, list or NumPy array) | `None` (no error) or the index 0, 1, 2 of the data qubit that was flipped | not exactly two integers equal to 0 or 1 |
| `correction_circuit(error_qubit)` | `None` or 0, 1, 2 | 3-qubit circuit that applies X to that qubit (nothing for `None`) | any other value |
| `add_feedforward_correction(qc, data, ancillas, syndrome)` | circuit `qc`, three data qubit indices, two ancilla indices, a 2-bit `ClassicalRegister` that is part of `qc` | modifies `qc` **in place** (return value ignored): syndrome extraction, measurement of the ancillas into `syndrome`, X corrections controlled by the register; touches no other qubit | invalid layout (use `validate_layout`) |
| *(bonus)* `phase_flip_encode_circuit()` | - | 3-qubit circuit mapping (α\|0⟩ + β\|1⟩) ⊗ \|00⟩ to α\|+++⟩ + β\|---⟩ | - |

### What this code can and cannot do (the documented limitations)
1. It corrects **at most one bit flip** per round. Two flips look like one flip of the *third* qubit: the decoder then adds a third flip and the logical qubit is **flipped** (a logical X error).
2. It does **not** protect against **phase flips** (Z errors): they change no parity, the syndrome stays (0, 0) and the logical phase is corrupted. (The bonus code swaps the roles.)
3. The syndrome extraction and the correction themselves are assumed **perfect** - this is a textbook model, not a fault-tolerant scheme.

## Concept: majority vote without looking
* **Classically** you protect a bit by writing it three times and taking a majority vote. **Quantum states cannot be copied** (no-cloning), but a CNOT spreads the information: α|0⟩ + β|1⟩ becomes α|000⟩ + β|111⟩ - the amplitudes α and β are shared by the three qubits.
* **Measuring the data qubits would destroy the superposition.** Instead we measure *parities*: "do qubit 0 and qubit 1 agree?" and "do qubit 1 and qubit 2 agree?". Both code words |000⟩ and |111⟩ give parity 0 for both questions, so the result tells **nothing** about α and β - but a single bit flip makes one or both parities 1. The pair of answers is the **syndrome**.
* **Feed-forward.** The syndrome is classical information; a gate that is *controlled by a classical bit* (`if_test`) can undo the error immediately. The exact simulator in this challenge follows every possible measurement outcome.
* **Limits.** See the three bullets above. Real codes (surface code, ...) fight both kinds of errors at the price of many more qubits.

In [ ]:
import numbers

import numpy as np
from qiskit import QuantumCircuit
from qiskit_fest.challenges.c09_error_correction import validate_layout

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit_fest.challenges.c09_error_correction import (
    ANCILLAS, DATA, LOGICAL_STATES, code_state, observed_syndrome, plot_recovery, run_recovery)
from qiskit_fest.viz import draw_circuit

alpha, beta = LOGICAL_STATES["generic"]
print("logical state alpha|0> + beta|1> with alpha = %.3f, beta = %.3f%+.3fi" % (alpha, beta.real, beta.imag))
print("its encoded version alpha|000> + beta|111> has amplitudes", np.round(code_state(alpha, beta).data[[0, 7]], 3), "on |000> and |111>")

## Part 1 · Encode  *(TODO)*
Complete `encode_circuit()`: a three-qubit circuit that turns (α|0⟩ + β|1⟩) ⊗ |00⟩ into α|000⟩ + β|111⟩. The logical qubit is qubit 0.

<details><summary>💡 Hint 1 (subtle)</summary>

Which two-qubit gate copies the *basis value* of one qubit onto another - and what does it do to a superposition?
</details>

<details><summary>💡 Hint 2</summary>

A CNOT with control qubit 0 and target qubit 1 maps |b⟩|0⟩ to |b⟩|b⟩ for b = 0, 1, and by linearity it also entangles superpositions. You need the value of qubit 0 on *both* other qubits.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`QuantumCircuit(3)`, then `cx(0, 1)` and `cx(0, 2)`. (`cx(0, 1)` followed by `cx(1, 2)` works as well.)
</details>

In [ ]:
def encode_circuit() -> QuantumCircuit:
    """3-qubit circuit: (alpha|0> + beta|1>) on qubit 0 and |00> on qubits 1, 2  ->  alpha|000> + beta|111>."""
    # TODO: spread the value of qubit 0 over qubits 1 and 2.
    raise NotImplementedError("TODO: implement encode_circuit()")

In [ ]:
with if_ready():
    draw_circuit(encode_circuit(), "Encoding")
    start = Statevector.from_label("00").tensor(Statevector([alpha, beta]))       # qubit 0 = (alpha, beta), qubits 1, 2 = |00>
    encoded = start.evolve(encode_circuit())
    print("overlap with alpha|000> + beta|111> =", round(abs(np.vdot(code_state(alpha, beta).data, encoded.data)), 6), "(1 = correct up to a global phase)")

In [ ]:
report = run_public_checks(9, globals(), keyword="encode_circuit")

## Part 2 · Extract the syndrome  *(TODO)*
Complete `syndrome_circuit()`: a **5-qubit** circuit (data qubits 0-2, ancilla qubits 3, 4 in |00⟩) that stores `s0` = parity of qubits 0 and 1 in qubit 3 and `s1` = parity of qubits 1 and 2 in qubit 4, **without changing the data qubits**. No measurement here - the measurement comes in Part 5.
The helper `observed_syndrome(syndrome_circuit, k)` runs your circuit on the code word |000⟩ with an X error on data qubit `k` (`None` = no error) and reads the two ancillas.

<details><summary>💡 Hint 1 (subtle)</summary>

A CNOT *adds* (XORs) the control bit into the target. What do you get when you XOR the bits of two data qubits into one ancilla?
</details>

<details><summary>💡 Hint 2</summary>

Ancilla 3 must end as q0 ⊕ q1: apply `cx(0, 3)` and `cx(1, 3)`. Ancilla 4 must end as q1 ⊕ q2. Data qubits are only ever *controls*, so they are never changed.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`QuantumCircuit(5)`; `cx(0, 3)`, `cx(1, 3)`, `cx(1, 4)`, `cx(2, 4)`.
</details>

In [ ]:
def syndrome_circuit() -> QuantumCircuit:
    """5-qubit circuit: ancilla 3 <- parity(q0, q1), ancilla 4 <- parity(q1, q2); the data qubits are unchanged. No measurement."""
    # TODO: CNOTs from the data qubits onto the ancillas.
    raise NotImplementedError("TODO: implement syndrome_circuit()")

In [ ]:
with if_ready():
    draw_circuit(syndrome_circuit(), "Syndrome extraction (no measurement yet)")
    print("error on data qubit ->  syndrome (s0, s1) read from the ancillas")
    for k in (None, 0, 1, 2):
        print(f"{str(k):>20s} -> {observed_syndrome(syndrome_circuit, k)}")

In [ ]:
report = run_public_checks(9, globals(), keyword="syndrome_circuit")

## Part 3 · Decode the syndrome  *(TODO)*
Complete `decode_syndrome(syndrome)`: from the pair `(s0, s1)` return the index (0, 1 or 2) of the flipped data qubit, or `None` for "no error". Read the table from the experiment above: which syndrome belongs to which flipped qubit? Reject anything that is not two bits (`ValueError`).

<details><summary>💡 Hint 1 (subtle)</summary>

Look at the printed table of Part 2 backwards: for every syndrome there is exactly one single-qubit explanation.
</details>

<details><summary>💡 Hint 2</summary>

Reason it out: a flip of qubit 0 changes *only* the parity of (0, 1); qubit 1 appears in *both* parities; qubit 2 only in the second one. A dictionary from the tuple `(s0, s1)` to the answer is enough.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Convert the input to a tuple of two ints and check each entry is an integer equal to 0 or 1; then `{(0, 0): None, (1, 0): ..., (1, 1): ..., (0, 1): ...}[syndrome]` with the three indices filled in from your table.
</details>

In [ ]:
def decode_syndrome(syndrome):
    """Index (0, 1, 2) of the single flipped data qubit for the syndrome (s0, s1), or None for no error. ValueError unless two 0/1 integers."""
    # TODO: validate the syndrome (ValueError) and look the answer up.
    raise NotImplementedError("TODO: implement decode_syndrome()")

In [ ]:
with if_ready():
    for k in (None, 0, 1, 2):
        s = observed_syndrome(syndrome_circuit, k)
        print(f"flip of {str(k):>4s}: syndrome {s} -> decoded as {decode_syndrome(s)}")

In [ ]:
report = run_public_checks(9, globals(), keyword="decode_syndrome")

## Part 4 · The correction  *(TODO)*
Complete `correction_circuit(error_qubit)`: a three-qubit circuit that applies X to the given data qubit (`0`, `1` or `2`), or does nothing for `None`. Reject every other value with `ValueError`.

<details><summary>💡 Hint</summary>

An X gate undoes an X error (X · X = I). Build `QuantumCircuit(3)` and add one `x` gate - or none.
</details>

In [ ]:
def correction_circuit(error_qubit) -> QuantumCircuit:
    """3-qubit circuit applying X to data qubit `error_qubit` (0, 1 or 2); an empty circuit for None. ValueError otherwise."""
    # TODO: validate error_qubit (ValueError), add the X gate.
    raise NotImplementedError("TODO: implement correction_circuit()")

In [ ]:
with if_ready():
    draw_circuit(correction_circuit(1), "Correction of an error on data qubit 1")

In [ ]:
report = run_public_checks(9, globals(), keyword="correction_circuit")

## Part 5 · The recovery step as a dynamic circuit  *(TODO)*
Complete `add_feedforward_correction(qc, data, ancillas, syndrome)`. It appends to `qc` (in place):
1. the syndrome extraction on `data` and `ancillas` (assumed to be in |00⟩),
2. the measurement of the ancillas into the register `syndrome` (`syndrome[0]` ← ancilla 0, `syndrome[1]` ← ancilla 1),
3. an X correction on the right data qubit, **controlled by the classical register** (`with qc.if_test((syndrome, value)):`).
Use the index lists you are given - the tests use many different qubit layouts - and call `validate_layout` first.

<details><summary>💡 Hint 1 (subtle)</summary>

You already have all the ingredients: a circuit that writes the syndrome, a function that says which qubit to flip and a circuit that flips it. What is missing is the *measurement* and the *classical condition*.
</details>

<details><summary>💡 Hint 2</summary>

`qc.compose(syndrome_circuit(), qubits=data + ancillas, inplace=True)` places your 5-qubit circuit on the given qubits (first the three data qubits, then the two ancillas). Then `qc.measure(ancillas[0], syndrome[0])` and the same for the second ancilla. The register holds `s0 + 2*s1`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

For each of the three single-flip syndromes `(s0, s1)` in `[(1, 0), (1, 1), (0, 1)]`: `with qc.if_test((syndrome, s0 + 2 * s1)):` and inside `qc.compose(correction_circuit(decode_syndrome((s0, s1))), qubits=data, inplace=True)`. The syndrome `(0, 0)` needs no gate.
</details>

In [ ]:
def add_feedforward_correction(qc: QuantumCircuit, data, ancillas, syndrome) -> None:
    """Append syndrome extraction, measurement into the 2-bit register `syndrome` and classically controlled X corrections to `qc` (in place)."""
    # TODO: validate_layout, compose syndrome_circuit, measure the ancillas, add the three if_test blocks.
    raise NotImplementedError("TODO: implement add_feedforward_correction()")

In [ ]:
with if_ready():
    scenarios = {"no error": [], "X on q0": [("X", 0)], "X on q1": [("X", 1)], "X on q2": [("X", 2)]}
    rows = {name: run_recovery(encode_circuit, add_feedforward_correction, (alpha, beta), errs)["min_fidelity"] for name, errs in scenarios.items()}
    print({k: round(v, 6) for k, v in rows.items()})
    plot_recovery(rows, "Single bit flips are repaired (generic logical state)")

In [ ]:
report = run_public_checks(9, globals(), keyword="add_feedforward_correction")

## Part 6 · Where the code gives up  *(not graded)*
Test the limits stated in the problem: two bit flips, a phase flip and a Y error (which is a bit flip **and** a phase flip).

In [ ]:
with if_ready():
    limits = {
        "no error": [], "X on q1": [("X", 1)],
        "X on q0 and q1 (two flips)": [("X", 0), ("X", 1)],
        "Z on q1 (phase flip)": [("Z", 1)],
        "Y on q1": [("Y", 1)],
    }
    rows = {name: run_recovery(encode_circuit, add_feedforward_correction, (alpha, beta), errs)["min_fidelity"] for name, errs in limits.items()}
    for name, value in rows.items():
        print(f"{name:30s} fidelity with the ideal code word: {value:.3f}")
    plot_recovery(rows, "What the bit-flip code can and cannot repair")

## Bonus · Protect against phase flips  *(optional)*
A phase flip Z becomes a bit flip X in the Hadamard basis (H Z H = X). Complete `phase_flip_encode_circuit()`: α|0⟩ + β|1⟩ ↦ α|+++⟩ + β|---⟩ on three qubits. Then Z errors on the code word can be repaired by the machinery you already have - *between two layers of Hadamards* (the helper does this when you pass `phase_flip=True`).

<details><summary>💡 Hint</summary>

Encode with the bit-flip code first, then rotate every qubit into the Hadamard basis.
</details>

In [ ]:
def phase_flip_encode_circuit() -> QuantumCircuit:
    """3-qubit circuit: (alpha|0> + beta|1>) on qubit 0 and |00> on qubits 1, 2  ->  alpha|+++> + beta|--->."""
    # TODO (bonus): bit-flip encoding followed by a basis change on every qubit.
    raise NotImplementedError("TODO (bonus): implement phase_flip_encode_circuit()")

In [ ]:
with if_ready():
    rows = {}
    for name, errs in {"no error": [], "Z on q0": [("Z", 0)], "Z on q1": [("Z", 1)], "Z on q2": [("Z", 2)], "X on q1 (not covered)": [("X", 1)]}.items():
        rows[name] = run_recovery(phase_flip_encode_circuit, add_feedforward_correction, (alpha, beta), errs, phase_flip=True)["min_fidelity"]
    print({k: round(v, 3) for k, v in rows.items()})
    plot_recovery(rows, "The phase-flip code repairs Z errors, but not X errors")

In [ ]:
report = run_public_checks(9, globals(), keyword="bonus")

## Reflection
1. Why does measuring the two parities not reveal α and β, and what would happen if you measured qubit 0 directly?
2. Show with a pencil why a double flip of qubits 0 and 1 produces the same syndrome as a single flip of qubit 2. What does the decoder do, and what is the final state?
3. If a bit flip has probability p on each qubit, the encoded qubit fails with probability about 3p². When is the code better than an unprotected qubit?
4. The ancilla measurement and the CNOTs can fail too. Which of the assumptions in the problem would a real quantum computer violate first?
5. What does the Y error do, and why can it be seen as a bit flip *and* a phase flip?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 9)